# **🎨 Chroma Database**

---
---
# Create from .txt file

- Creates guidelines.txt with multiple lines.
- Reads lines from the file.
- Generates unique IDs for each line.
- Inserts them into a Chroma collection with optional metadata (line number).

In [ ]:
import chromadb
import uuid

# Initialize Chroma client
client = chromadb.Client()

# Create or get collection
collection = client.get_or_create_collection(name="guidelines_collection")

# Read the guidelines file
with open("guidelines.txt", "r", encoding="utf-8") as f:
    guidelines = f.read().splitlines()

# Upsert into Chroma (use upsert to avoid duplicates)
collection.upsert(
    ids=[str(uuid.uuid4()) for _ in guidelines],
    documents=guidelines,
    metadatas=[{"line": i} for i in range(len(guidelines))]
)

print("✅ Guidelines added to Chroma collection!")

# Optional: peek at the first few entries
print(collection.peek())

In [ ]:
query_text = "sports"
results = collection.query(
    query_texts=[query_text],
    n_results=3  # return top 3 closest results
)

print(f"\nQuery: {query_text}\nTop matches:")
for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"- {doc} (distance={distance:.4f})")

---
---
# Simple Implementation

In [ ]:
import chromadb
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

chroma_client = chromadb.Client()

# switch `create_collection` to `get_or_create_collection` to avoid creating a new collection every time
collection = chroma_client.get_or_create_collection(name="my_collection")

# switch `add` to `upsert` to avoid adding the same documents every time
collection.upsert(
    documents=[
        "Pineapples are tropical fruits with a spiky skin and sweet yellow flesh, rich in vitamin C.",
        "Oranges are citrus fruits known for their tangy flavor and high vitamin C content.",
        "Bananas are elongated yellow fruits that provide potassium and quick energy.",
        "Apples are crisp fruits that come in red, green, and yellow varieties, often eaten raw or baked in pies.",
        "Mangoes are tropical stone fruits with juicy orange flesh, often used in smoothies and desserts.",
        "Strawberries are small red berries with tiny seeds on the outside, commonly used in jams and cakes.",
        "Lemons are sour citrus fruits with bright yellow skin, often used to make lemonade.",
        "Blueberries are small round fruits with blue-purple skin, packed with antioxidants."
    ],
    ids=[f"id{i}" for i in range(1, 9)]
)

# 2. Retrieve embeddings from collection
# Chroma stores embeddings internally; we can fetch them along with docs
all_docs = collection.get(include=["documents", "embeddings"])
documents = all_docs['documents']
embeddings = all_docs['embeddings']

# Convert embeddings to a tensor or numpy array
import numpy as np
embeddings = np.array(embeddings)

# 3. Reduce to 2D using PCA
pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)

results = collection.query(
    query_texts=["workout"], # Chroma will embed this for you
    n_results=2 # how many results to return
)

print(results)
# 4. Plot documents in 2D
plt.figure(figsize=(10, 7))
plt.scatter(reduced[:,0], reduced[:,1], c='blue', s=80)

for i, doc in enumerate(documents):
    plt.annotate(doc[:30]+"..." if len(doc) > 30 else doc, 
                 (reduced[i,0]+0.01, reduced[i,1]+0.01))

plt.title("Chroma Embeddings Visualization (PCA 2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

# Basic Implementation from documentation
- https://docs.trychroma.com/docs/overview/getting-started

In [ ]:
import chromadb
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(name="my_collection")

collection.add(
    ids=["id1", "id2"],
    documents=[
        "This is a document about pineapple",
        "This is a document about oranges"
    ]
)

results = collection.query(
    query_texts=["This is a query document about hawaii"], # Chroma will embed this for you
    n_results=2 # how many results to return
)
print(results)

In [ ]:
import chromadb
chroma_client = chromadb.Client()

# switch `create_collection` to `get_or_create_collection` to avoid creating a new collection every time
collection = chroma_client.get_or_create_collection(name="my_collection")

# switch `add` to `upsert` to avoid adding the same documents every time
collection.upsert(
    documents=[
        "This is a document about pineapple",
        "This is a document about oranges"
    ],
    ids=["id1", "id2"]
)

results = collection.query(
    query_texts=["This is a query document about florida"], # Chroma will embed this for you
    n_results=2 # how many results to return
)

print(results)


In [ ]:
import chromadb
from chromadb.config import Settings

# durable mode (saved to folder "chroma_db")
client = chromadb.PersistentClient(path="chroma_db")

collection = client.get_or_create_collection("my_collection")
collection.upsert(
    ids=["1"],
    documents=["This is durable data"],
    metadatas=[{"topic": "durability"}]
)

# later (in another script), you can reload it:
client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_collection("my_collection")
print(collection.peek())  # ✅ still there

---
---
---